In [1]:
!wget -q https://github.com/karoldvl/ESC-50/archive/master.zip
!unzip -q master.zip

In [2]:
import pandas as pd

df = pd.read_csv("ESC-50-master/meta/esc50.csv")
df.head()



,filename,fold,target,category,esc10,src_file,take
0,1-100032-A-0.wav,1,0,dog,True,100032,A
1,1-100038-A-14.wav,1,14,chirping_birds,False,100038,A
2,1-100210-A-36.wav,1,36,vacuum_cleaner,False,100210,A
3,1-100210-B-36.wav,1,36,vacuum_cleaner,False,100210,B
4,1-101296-A-19.wav,1,19,thunderstorm,False,101296,A


In [3]:
alerts = ["door_wood_knock", "clock_alarm", "car_horn", "siren"]
background = ["clock_tick", "door_wood_creaks", "footsteps", "keyboard_typing",
              "mouse_click", "breathing", "coughing", "rain",
              "pouring_water", "drinking_sipping", "washing_machine", "vacuum_cleaner"]

df = df[df["category"].isin(alerts + background)]
df.loc[df["category"].isin(background), "category"] = "background"

df["category"].value_counts()

,count
category,
background,480
door_wood_knock,40
clock_alarm,40
car_horn,40
siren,40


In [4]:
from IPython.display import Audio

knock = df[df["category"] == "background"].iloc[9]
Audio("ESC-50-master/audio/" + knock["filename"])

In [5]:
train_df = df[df["fold"].isin([1, 2, 3])]
val_df = df[df["fold"] == 4]
test_df = df[df["fold"] == 5]

print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))

train: 384
val: 128
test: 128


In [7]:
import librosa
import numpy as np

first_file = train_df["filename"].iloc[0]
audio, sr = librosa.load("ESC-50-master/audio/" + first_file, sr=16000)

print(first_file)
print(audio.shape)
print(audio[:10])

1-100210-A-36.wav
(80000,)
[-0.00718514 -0.00815905 -0.00541599 -0.00315152 -0.00759226 -0.00907682
 -0.01746516 -0.00673746  0.00370465 -0.00655121]


In [8]:
mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
print(mfcc.shape)

(20, 157)


In [9]:
means = mfcc.mean(axis=1)
spreads = mfcc.std(axis=1)
features = np.concatenate([means, spreads])

print(means.shape, spreads.shape, features.shape)

(20,) (20,) (40,)


In [10]:
def get_mfcc(filename):
    audio, sr = librosa.load("ESC-50-master/audio/" + filename, sr=16000)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])

test = get_mfcc(train_df["filename"].iloc[0])
print(test.shape)

(40,)


In [12]:
X_train = np.array([get_mfcc(f) for f in train_df["filename"]])
X_val = np.array([get_mfcc(f) for f in val_df["filename"]])
X_test = np.array([get_mfcc(f) for f in test_df["filename"]])

y_train = train_df["category"].values
y_val = val_df["category"].values
y_test = test_df["category"].values

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)

(384, 40) (128, 40) (128, 40)
(384,) (128,) (128,)
